# F02-P2 General

**Site Characterization: General Context, components 1.1 to 1.7.**

Overlays the AOI with pre-defined layers using simple operations: clip, mask, count, area.
This is the first module a user sees. It describes where the site is, what it sits on, what
has happened to it, and what threatens it.

> **Not runnable yet.** All analysis logic below is real Python. Only file access is stubbed.
> The four stubs in `common.py` raise `NotImplementedError`. Filling them is what makes this
> notebook run; nothing else needs rewriting.

## Conventions that hold across the whole notebook

- The AOI is heterogeneous. It spans many pixels and can cover several categories at once, so
  outputs are area weighted distributions, not single labels.
- Every component returns a `ComponentResult` with `tables` for the frontend charts, `values`
  for downstream notebooks, and one `narrative` string.
- Unresolved logic is marked with a `flags` entry, never silently skipped.
- The AOI is reprojected to ESRI:54034 once, in Setup. No component reprojects it again.

## Handoff

This notebook writes `outputs/<aoi_id>__F02-P2-general.json`. `F02-P2 Nature`, `F02-P3 Threats`
and `F02-P4 Pathway` read it back with `load_results`.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import math
from dataclasses import dataclass

import geopandas as gpd
import numpy as np

# House style, same as sea_deforestation_risk: config and common are star imported so the
# component cells stay close to the logic.
from config import *
from common import *

### AOI input

`aoi_id` names the run. It becomes the prefix of every result file, so keep it stable across
notebooks for one project area.

In [2]:
aoi_id = AOI_ID   # AOI is set in config.py (AOI_PATH, AOI_ID); change it there, then restart the kernel

aoi = prepare_aoi(gpd.read_file(AOI_PATH))
print(f"AOI {aoi_id}: {fmt_ha(aoi.area_ha)}, supplied in {aoi.source_crs}, "
      f"measured in {REFERENCE_CRS}")

results: dict[str, ComponentResult] = {}

AOI aoi1: 67,439 ha, supplied in EPSG:4326, measured in ESRI:54034


---
## 1.1 Ecosystem Type

Reports the ecosystem setting of the AOI as area and share per ecosystem, for a pie chart, plus
a composition narrative. The result is also the Axis 3 reference ecosystem that drives
downstream logic.

**Data.** Derived from the pathway raster's ecosystem band (band 2), not a separate layer.
Band-2 codes are remapped to three Axis 3 classes: dryland forest (1) and savanna (4) both
become Dryland, mangrove (2) and peatland (3) keep their own class, and 0 (none) falls into
Other. Merging dryland forest and savanna is the team's choice.

**Decisions locked.**

- Percentage denominator = total AOI area. An "Other/Unclassified" slice absorbs nodata and
  non ecosystem pixels so the pie sums to 100.
- Composition uses pure presence. A class counts as present if its area is above zero, with no
  threshold. Open risk: a single stray edge pixel can flip "single" to "combination".

**Open item.** Three classes give 2^3 = 8 subsets, and the spec listed only the 7 non empty
ones. An AOI that is entirely water or unclassified land falls into the eighth case. It is
handled below with a default narrative and an `UNRESOLVED` flag. Team decision needed: reject
such an AOI, or continue with reduced output. Without Axis 3 the pathway module cannot run.

**Example render.**

> This area sits on peat, a soil built up from layers of organic material that stays wet for
> most of the year. Its natural reference ecosystem is peatland, a waterlogged system shaped by
> its deep organic soil.

**Downstream use.** `present_set` is the Axis 3 reference ecosystem. It drives the Cat 8
ecosystem conditional in the pathway module and the Ecosystem Applicability filter in the
Activity Catalog.

In [3]:
DRYLAND, MANGROVE, PEATLAND = 1, 2, 3

ECOSYSTEM_NARRATIVE: dict[frozenset[int], str] = {
    frozenset({DRYLAND}): (
        "This area sits on mineral soil that is not regularly flooded, unlike peatland or "
        "mangrove. Its natural reference ecosystem is dryland forest, dominated by trees "
        "growing on well-drained land."
    ),
    frozenset({MANGROVE}): (
        "This area sits in a coastal zone with salt or brackish water shaped by the tides. Its "
        "natural reference ecosystem is mangrove, made up of trees and shrubs adapted to "
        "waterlogged, salty ground."
    ),
    frozenset({PEATLAND}): (
        "This area sits on peat, a soil built up from layers of organic material that stays wet "
        "for most of the year. Its natural reference ecosystem is peatland, a waterlogged "
        "system shaped by its deep organic soil."
    ),
    frozenset({MANGROVE, PEATLAND}): (
        "This area is a coastal zone with tidal salt or brackish water that also sits on peat "
        "soil. It combines mangrove and peatland features in the same place."
    ),
    frozenset({DRYLAND, PEATLAND}): (
        "This area includes both mineral-soil ground and peat soil. Part of it follows a "
        "dryland reference, and part follows a peatland reference shaped by its wet, organic "
        "soil."
    ),
    frozenset({DRYLAND, MANGROVE}): (
        "This area spans both non-flooded mineral soil inland and a tidal coastal zone. It "
        "combines dryland and mangrove references across different parts of the site."
    ),
    frozenset({DRYLAND, MANGROVE, PEATLAND}): (
        "This area spans all three settings: non-flooded mineral soil inland, a tidal coastal "
        "zone with salt or brackish water, and ground built on wet peat soil. It combines "
        "dryland, mangrove, and peatland references across different parts of the site."
    ),
    # Eighth case, see the Open item above.
    frozenset(): (
        "This area does not fall within any mapped ecosystem type. It may be open water or "
        "land outside the mapped extent. No reference ecosystem can be assigned."
    ),
}


def analyze_ecosystem_type(aoi: AOI) -> ComponentResult:
    """Component 1.1. Ecosystem composition and the Axis 3 reference ecosystem."""
    # Ecosystem is derived from the pathway raster's ecosystem band (band 2), not a separate
    # layer, and remapped to the 3-class Axis 3 scheme: dryland forest (1) and savanna (4) both
    # become Dryland. Denominator is the whole site, so the pie can carry an Other slice.
    band = load_raster_clipped(PATHWAY_RASTER, aoi, resampling="nearest",
                               band=PATHWAY_ECOSYSTEM_BAND)
    vals = band.values.filled(-1)
    rows: list[ClassShare] = []
    for code, label in ECOSYSTEM_CLASSES.items():
        src_codes = [s for s, d in PATHWAY_ECO_TO_AXIS3.items() if d == code]
        area_ha = int(np.isin(vals, src_codes).sum()) * band.pixel_area_ha
        rows.append(ClassShare(code=code, label=label, area_ha=area_ha,
                               pct=safe_pct(area_ha, aoi.area_ha)))

    mapped_ha = sum(r.area_ha for r in rows)
    other_ha = max(0.0, aoi.area_ha - mapped_ha)
    rows.append(
        ClassShare(
            code="other",
            label="Other/Unclassified",
            area_ha=other_ha,
            pct=safe_pct(other_ha, aoi.area_ha),
        )
    )

    # Derived output raster: the 3-class Axis 3 ecosystem grid.
    eco_codes = np.zeros(band.values.shape, dtype="float32")
    for _src, _dst in PATHWAY_ECO_TO_AXIS3.items():
        eco_codes[vals == _src] = _dst
    eco_slice = RasterSlice(
        np.ma.masked_array(eco_codes, mask=np.ma.getmaskarray(band.values)),
        band.pixel_area_ha, band.transform, band.crs,
    )

    # Pure presence. "Other" is never part of present_set; it does not drive ecosystem logic.
    present_set = frozenset(r.code for r in rows if r.code != "other" and r.area_ha > 0)

    flags: list[str] = []
    if not present_set:
        flags.append(
            "UNRESOLVED 1.1: AOI has no mapped ecosystem. Downstream modules that require an "
            "Axis 3 reference ecosystem cannot run."
        )

    return ComponentResult(
        component="1.1 Ecosystem Type",
        applicable=bool(present_set),
        narrative=ECOSYSTEM_NARRATIVE[present_set],
        tables={"ecosystem_composition": rows},  # pie chart, sums to 100
        values={"present_set": present_set, "mapped_ha": mapped_ha, "other_ha": other_ha},
        rasters={"1.1_ecosystem_class": eco_slice},
        flags=flags,
    )


results["1.1"] = analyze_ecosystem_type(aoi)
show_result(results["1.1"])

[1.1 Ecosystem Type]
  This area sits on mineral soil that is not regularly flooded, unlike peatland or mangrove. Its natural reference ecosystem is dryland forest, dominated by trees growing on well-drained land.
  ecosystem_composition:


,code,label,area_ha,pct
0,1,Dryland,66600.728308,98.756962
1,2,Mangrove,0.000000,0.000000
2,3,Peatland,0.000000,0.000000
3,other,Other/Unclassified,838.292822,1.243038


  saved table: D:\NBSTOOLV3\OUTPUTS\aoi1\tables\1.1_ecosystem_composition.csv


{'present_set': frozenset({1}),
 'mapped_ha': 66600.72830803317,
 'other_ha': 838.2928220217291}

  saved raster: D:\NBSTOOLV3\OUTPUTS\aoi1\rasters\1.1_ecosystem_class.tif


---
## 1.2 Administrative Boundaries

Reports where the project area sits administratively, leading with the district and province
that hold most of it, and lists every other district it touches.

**Data.** GADM v4.1. L0 = country, L1 = province, L2 = district.

**Decisions locked.**

- Sliver threshold: a unit is reported only if its intersection is at least 1% of the AOI. This
  removes false slivers from the generalised GADM boundary lines. Stricter than the pure
  presence rule in 1.1 on purpose, because boundary geometry is coarser than the raster.
- The narrative names the dominant district and its province, and quotes the total AOI area, not
  the area inside that district.
- The province in the sentence is read from the dominant district's own `NAME_1`, not from the
  largest province. Those two can differ: an AOI can be 60% in province A split across three
  small districts and 40% in province B as one large district. Taking the parent of the named
  district keeps the pair internally consistent.
- The country is not narrated, but L0 is still processed, because `dominant_country` drives the
  national risk comparison in 1.6.
- Term for GADM L2 is "district" throughout, narrative and table header. L3 (kecamatan, huyen,
  amphoe) is not used by the tool.

**Known limitation, accepted by the team.** The second sentence always says "Within this
province". For an AOI that spans more than one province that phrasing is factually wrong, and
for a transboundary AOI it also hides the second country. The component raises a `flags` entry
in both cases so the mismatch is recorded in the output rather than passing silently.

**Example render, single district.**

> This project area is majorly located in Pelalawan, Riau with an approximate total area of
> 1,240 hectares.

**Example render, multiple districts.**

> This project area is majorly located in Pelalawan, Riau with an approximate total area of
> 1,240 hectares. Within this province, it also overlaps with the following districts:

| District | Province | Area (ha) |
|---|---|---|
| Pelalawan | Riau | 620 |
| Indragiri Hulu | Riau | 280 |
| Tebo | Jambi | 340 |

**Downstream use.** The country result gates which national datasets and policies apply, for
example the APD land status overlay, which is Indonesia first. The province result links to
`gadm41_L1_with_region.csv` for the deforestation risk regions. L2 availability varies by
country, so the component falls back to province level when no district is returned.

In [4]:
@dataclass(frozen=True)
class AdminUnit:
    name: str
    area_ha: float
    pct: float
    parent: str | None = None  # province for a district, country for a province


def _admin_units(gdf, aoi: AOI, group_cols: list[str], name_field: str,
                 parent_field: str | None = None) -> list[AdminUnit]:
    """One admin level from the combined boundary layer, dissolved by name.

    The single shapefile is district-level, so a country or province appears as many rows. We
    dissolve by the level's NAME columns (COUNTRY, NAME_1, NAME_2), not the GID columns, because
    GID_1/GID_2 are blank for Indonesia in this file and grouping on a blank key drops the unit.
    Ancestor names are included so same-named units in different parents stay separate. Rows with
    a blank key are dropped first, for the same reason.
    """
    if gdf.empty:
        return []
    sub = gdf
    for col in group_cols:
        sub = sub[sub[col].notna() & (sub[col].astype(str).str.strip() != "")]
    if sub.empty:
        return []
    aoi_geom = aoi.geometry.iloc[0]
    diss = sub.dissolve(by=group_cols, as_index=False)
    units = []
    for _, row in diss.iterrows():
        area_ha = row.geometry.intersection(aoi_geom).area / M2_PER_HA
        units.append(AdminUnit(
            name=str(row[name_field]),
            area_ha=area_ha,
            pct=safe_pct(area_ha, aoi.area_ha),
            parent=str(row[parent_field]) if parent_field else None,
        ))
    kept = [u for u in units if u.pct >= ADMIN_SLIVER_PCT]
    return sort_by_area(kept)


def analyze_admin_boundaries(aoi: AOI) -> ComponentResult:
    """Component 1.2. Where the project area sits administratively."""
    gdf = load_vector_intersecting(ADMIN_BOUNDARIES, aoi)
    countries = _admin_units(gdf, aoi, *ADMIN_LEVELS["country"])
    provinces = _admin_units(gdf, aoi, *ADMIN_LEVELS["province"])
    districts = _admin_units(gdf, aoi, *ADMIN_LEVELS["district"])

    # "approximate total area" is the whole AOI, not the part inside the named district.
    area_text = f"{aoi.area_ha:,.0f}"

    flags: list[str] = []

    if districts:
        main = districts[0]
        # Province of the named district, not the largest province. See the note above.
        main_province = main.parent or (provinces[0].name if provinces else None)
        where = f"{main.name}, {main_province}" if main_province else main.name
    elif provinces:
        # L2 is missing for some countries. Fall back to province level rather than emit a
        # sentence with an empty slot.
        where = provinces[0].name
        flags.append(
            "1.2: no GADM L2 district returned above the 1% sliver threshold. The narrative "
            "falls back to province level."
        )
    else:
        where = None
        flags.append(
            "1.2: AOI does not intersect any GADM unit above the 1% sliver threshold. "
            "The national comparison in 1.6 will report as not applicable."
        )

    opening = (
        f"This project area is majorly located in {where} with an approximate total area of "
        f"{area_text} hectares."
        if where
        else f"This project area has an approximate total area of {area_text} hectares."
    )

    # Second sentence only when there is more than one district to list.
    follow = (
        "Within this province, it also overlaps with the following districts:"
        if len(districts) > 1
        else ""
    )

    if len(provinces) > 1:
        flags.append(
            f"1.2: AOI spans {len(provinces)} provinces, but the narrative says \"Within this "
            "province\". Accepted by the team; recorded here so the mismatch is visible."
        )
    if len(countries) > 1:
        flags.append(
            "1.2: AOI is transboundary. The narrative does not mention it, and the national "
            "risk comparison in 1.6 uses the dominant country only."
        )

    return ComponentResult(
        component="1.2 Administrative Boundaries",
        applicable=bool(countries or provinces or districts),
        narrative=sentences(opening, follow),
        tables={
            # Rendered under the narrative as District / Province / Area (ha), dominant first.
            # Every district is listed, including the one named in the sentence, so the areas
            # in the table add up to the AOI.
            "district_table": districts,
            "country": countries,
            "province": provinces,
        },
        values={
            "dominant_country": countries[0].name if countries else None,
            "dominant_province": districts[0].parent if districts else (
                provinces[0].name if provinces else None
            ),
            "dominant_district": districts[0].name if districts else None,
            "transboundary": len(countries) > 1,
            "provinces": [u.name for u in provinces],
        },
        flags=flags,
    )


results["1.2"] = analyze_admin_boundaries(aoi)
show_result(results["1.2"])

[1.2 Administrative Boundaries]
  This project area is majorly located in KAB. SANGGAU, KALIMANTAN BARAT with an approximate total area of 67,439 hectares. Within this province, it also overlaps with the following districts:
  district_table:


,name,area_ha,pct,parent
0,KAB. SANGGAU,48358.183106,71.706532,KALIMANTAN BARAT
1,KAB. LANDAK,19080.838024,28.293468,KALIMANTAN BARAT


  saved table: D:\NBSTOOLV3\OUTPUTS\aoi1\tables\1.2_district_table.csv
  country:


,name,area_ha,pct,parent
0,INDONESIA,67439.02113,100.0,None


  saved table: D:\NBSTOOLV3\OUTPUTS\aoi1\tables\1.2_country.csv
  province:


,name,area_ha,pct,parent
0,KALIMANTAN BARAT,67439.02113,100.0,INDONESIA


  saved table: D:\NBSTOOLV3\OUTPUTS\aoi1\tables\1.2_province.csv


{'dominant_country': 'INDONESIA',
 'dominant_province': 'KALIMANTAN BARAT',
 'dominant_district': 'KAB. SANGGAU',
 'transboundary': False,
 'provinces': ['KALIMANTAN BARAT']}

---
## 1.3 Protected Areas (WDPA)

Reports how much of the AOI overlaps protected areas and what those areas are designated for.
This is a key eligibility and additionality signal.

**Data.** `WDPA_polygon_4326.shp`, the same source as the backend `prep_wdpa_by_region`.

**Decisions locked.**

- Headline overlap uses the union of protected areas, because WDPA sites overlap each other and
  summing per site can exceed the AOI.
- No sliver threshold. Even a small overlap is legally meaningful.
- Filter: `STATUS` in {Designated, Inscribed, Established}; drop pure marine (`REALM == 'Marine'`), keep coastal for mangrove; polygon features only.
- The narrative names the designation type only, from `DESIG_ENG`. Site name, IUCN category and
  status are dropped from the prose. They stay in the per site table.
- Duplicate designation types are collapsed. Two national parks read as "National Park" once,
  ordered by total overlap area.
- The narrative opens with "Besides", so it assumes the frontend renders 1.2 and 1.3 as
  continuous prose, in that order.
- No percentage in the prose. Hectares only, matching the wording in 1.2.

**Known limitation, accepted by the team.** When the AOI does not overlap any protected area the
component emits an empty narrative, so nothing is rendered. This conflicts with the locked
decision in 1.7, which shows all five hazard cards precisely so that absence stays visible. The
practical cost here is larger than in 1.7: no overlap is a positive additionality argument, and
a missing card cannot be told apart from a WDPA layer that failed to load. The structured
`values` are still emitted (`protected_ha = 0`, `in_strict_pa = False`), so the pathway module
keeps its signal even with no prose.

**Example render, overlap.**

> Besides, this project area overlaps with 320 hectares of protected areas. The protected area
> within the polygon is designated for National Park.

**Example render, overlap with several designation types.**

> Besides, this project area overlaps with 320 hectares of protected areas. The protected area
> within the polygon is designated for National Park and Wildlife Reserve.

**Example render, no overlap.**

> (nothing)

**Downstream use.** A project inside a strict protected area (IUCN Ia, Ib, II) is hard to
justify on additionality, while overlap with unprotected land can strengthen the PROTECT
pathway. `in_strict_pa` carries that flag even though IUCN category is no longer narrated.

In [5]:
WDPA_KEEP_STATUS = {"Designated", "Inscribed", "Established"}
WDPA_DROP_REALM = "Marine"  # this extract has REALM (Terrestrial/Coastal/Marine), not MARINE
WDPA_STRICT_IUCN = {"Ia", "Ib", "II"}
WDPA_NO_DESIG = {"", "none", "not reported", "not applicable", "nan"}


@dataclass(frozen=True)
class ProtectedSite:
    name: str
    designation: str
    iucn_category: str
    status: str
    area_ha: float


def _unique_designations(sites: list[ProtectedSite]) -> list[str]:
    """Distinct DESIG_ENG values, ordered by total overlap area descending.

    Two national parks in one AOI should read as "National Park" once, not twice. Ordering by
    summed area rather than first appearance keeps the dominant designation first even when a
    tiny site of another type happens to sort earlier.
    """
    totals: dict[str, float] = {}
    for s in sites:
        if s.designation.strip().lower() in WDPA_NO_DESIG:
            continue  # a site with no usable designation cannot fill the slot
        totals[s.designation] = totals.get(s.designation, 0.0) + s.area_ha
    return sorted(totals, key=totals.get, reverse=True)


def analyze_protected_areas(aoi: AOI) -> ComponentResult:
    """Component 1.3. Legal protection status of the site."""
    gdf = load_vector_intersecting(WDPA_POLYGON, aoi)

    if not gdf.empty:
        gdf = gdf[gdf["STATUS"].isin(WDPA_KEEP_STATUS)]
        gdf = gdf[gdf["REALM"].astype(str).str.strip() != WDPA_DROP_REALM]
        gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]

    if gdf.empty:
        # Team decision: render nothing when there is no overlap. Values are still emitted so
        # the pathway module keeps the additionality signal. See the note above.
        return ComponentResult(
            component="1.3 Protected Areas",
            applicable=False,
            narrative="",
            tables={"sites": []},
            values={"protected_ha": 0.0, "protected_pct": 0.0, "in_strict_pa": False},
        )

    protected_ha = union_overlap_ha(aoi, gdf)
    protected_pct = safe_pct(protected_ha, aoi.area_ha)  # kept for the frontend, not narrated

    areas = per_feature_overlap_ha(aoi, gdf)
    sites = sort_by_area([
        ProtectedSite(
            name=str(gdf.iloc[i].get("NAME", "Unnamed site")),
            designation=str(gdf.iloc[i].get("DESIG_ENG", "Not Reported")),
            iucn_category=str(gdf.iloc[i].get("IUCN_CAT", "Not Reported")),
            status=str(gdf.iloc[i]["STATUS"]),
            area_ha=float(areas[i]),
        )
        for i in range(len(gdf))
    ])

    overlap_sentence = (
        f"Besides, this project area overlaps with {protected_ha:,.0f} hectares of "
        "protected areas."
    )

    flags: list[str] = []
    designations = _unique_designations(sites)
    if designations:
        designation_sentence = (
            "The protected area within the polygon is designated for "
            f"{oxford_join(designations)}."
        )
    else:
        # Every overlapping site lacks a usable DESIG_ENG. Drop the second sentence rather than
        # print "designated for Not Reported".
        designation_sentence = ""
        flags.append(
            "1.3: overlapping WDPA sites carry no usable DESIG_ENG. The designation sentence "
            "is omitted."
        )

    # Additionality constraint, read by the pathway module even though it is no longer narrated.
    in_strict_pa = any(s.iucn_category in WDPA_STRICT_IUCN for s in sites)

    return ComponentResult(
        component="1.3 Protected Areas",
        applicable=True,
        narrative=sentences(overlap_sentence, designation_sentence),
        tables={"sites": sites},
        values={
            "protected_ha": protected_ha,
            "protected_pct": protected_pct,
            "in_strict_pa": in_strict_pa,
            "designations": designations,
        },
        flags=flags,
    )


results["1.3"] = analyze_protected_areas(aoi)
show_result(results["1.3"])

[1.3 Protected Areas]
  Besides, this project area overlaps with 6,677 hectares of protected areas. The protected area within the polygon is designated for Nature Reserve.
  sites:


,name,designation,iucn_category,status,area_ha
0,Gunung Nyiut Penrissen,Nature Reserve,Ia,Designated,6676.955056


  saved table: D:\NBSTOOLV3\OUTPUTS\aoi1\tables\1.3_sites.csv


{'protected_ha': 6676.955056131181,
 'protected_pct': 9.900729494953369,
 'in_strict_pa': True,
 'designations': ['Nature Reserve']}

---
## 1.4 Terrain (Slope and Elevation)

Reports slope and elevation as two distributions, each a bar chart, plus a short narrative.

**Data.** One continuous elevation raster in metres (`ELEVATION_RASTER`). Slope is derived from
it with `slope_percent_from_dem` (gradient in percent, with an equal-area distance correction by
AOI latitude, since the reference CRS distorts distance). Both are then binned with
`ELEVATION_BREAKS` and `SLOPE_BREAKS`; the inputs are not pre-classified.

| Slope code | Label | Range | | Elevation code | Label | Range |
|---|---|---|---|---|---|---|
| 1 | Flat | 0-8% | | 1 | Lowland | 0-500 m |
| 2 | Gently sloping | 8-15% | | 2 | Submontane / hill | 500-1000 m |
| 3 | Moderately steep | 15-25% | | 3 | Montane | 1000-2000 m |
| 4 | Steep | 25-40% | | 4 | Upper montane | >2000 m |
| 5 | Very steep | >40% | | | | |

**Decisions locked.**

- Input is already binned, so the tool does no reclassification.
- Denominator = valid (non nodata) area, so each bar chart sums to 100.
- The elevation narrative reports exact min-max metres from the continuous raster; slope
  stays class based. Both tables stay class based.

**Example render.**

> Elevation ranges from 12 to 1840 m above sea level (asl), predominantly Lowland. Slopes are
> predominantly Gently sloping.

**Downstream use.** Terrain is a design constraint, not only a description. Steep slopes limit
some activities and raise erosion risk; elevation guides species and forest type choice.

In [6]:
def analyze_terrain(aoi: AOI) -> ComponentResult:
    """Component 1.4. Slope and elevation profile."""
    # Only elevation is read (continuous, metres). Slope is DERIVED from it, because a standalone
    # slope raster was not available. Both are then binned into class codes.
    elev_c = load_raster_clipped(ELEVATION_RASTER, aoi, resampling="nearest")
    if elev_c.valid_area_ha <= 0:
        return not_applicable(
            "1.4 Terrain", "No elevation data is available for this project area."
        )

    slope_c = slope_percent_from_dem(elev_c, aoi)
    slope = classify_continuous(slope_c, SLOPE_BREAKS)
    elev = classify_continuous(elev_c, ELEVATION_BREAKS)

    slope_rows = tabulate_classes(slope, SLOPE_CLASSES, denominator_ha=slope.valid_area_ha)
    elev_rows = tabulate_classes(elev, ELEVATION_CLASSES, denominator_ha=elev.valid_area_ha)

    dom_slope = dominant(slope_rows)
    dom_elev = dominant(elev_rows)

    elev_clause = ""
    if dom_elev:
        # Min and max in actual metres, from the continuous elevation raster (the class raster
        # cannot give metres). elev_c.values is masked, so min and max ignore nodata.
        vmin = int(round(float(elev_c.values.min())))
        vmax = int(round(float(elev_c.values.max())))
        # Flat AOI (a single elevation value): avoid "ranges from 12 to 12 m".
        elev_clause = (
            f"Elevation is {vmin} m above sea level (asl), predominantly {dom_elev.label}."
            if vmin == vmax
            else f"Elevation ranges from {vmin} to {vmax} m above sea level (asl), "
                 f"predominantly {dom_elev.label}."
        )
    slope_clause = f"Slopes are predominantly {dom_slope.label}." if dom_slope else ""

    return ComponentResult(
        component="1.4 Terrain",
        applicable=True,
        narrative=sentences(elev_clause, slope_clause),
        tables={"slope": slope_rows, "elevation": elev_rows},
        values={
            "dominant_slope": dom_slope.code if dom_slope else None,
            "dominant_elevation": dom_elev.code if dom_elev else None,
        },
        rasters={"1.4_slope_class": slope, "1.4_elevation_class": elev},
    )


results["1.4"] = analyze_terrain(aoi)
show_result(results["1.4"])

[1.4 Terrain]
  Elevation ranges from 42 to 1408 m above sea level (asl), predominantly Lowland. Slopes are predominantly Flat.
  slope:


,code,label,area_ha,pct
0,1,Flat,16228.08,24.168728
1,2,Gently sloping,15738.21,23.439157
2,3,Moderately steep,15087.51,22.470059
3,4,Steep,11446.65,17.047671
4,5,Very steep,8644.50,12.874386


  saved table: D:\NBSTOOLV3\OUTPUTS\aoi1\tables\1.4_slope.csv
  elevation:


,code,label,area_ha,pct
0,1,Lowland,61408.62,91.13735
1,2,Submontane / hill,5492.97,8.15219
2,3,Montane,478.71,0.71046
3,4,Upper montane,0.00,0.00000


  saved table: D:\NBSTOOLV3\OUTPUTS\aoi1\tables\1.4_elevation.csv


{'dominant_slope': 1, 'dominant_elevation': 1}

  saved raster: D:\NBSTOOLV3\OUTPUTS\aoi1\rasters\1.4_slope_class.tif
  saved raster: D:\NBSTOOLV3\OUTPUTS\aoi1\rasters\1.4_elevation_class.tif


---
## 1.5 Historical Deforestation (2014 to 2024)

Reports how much forest the AOI lost between 2014 and 2024 and the annual rate using Puyravaud.

**Data.** `FC2014.tif` (binary forest cover 2014, Tier 1-2) and `LC2024.tif` (20 class land cover
2024). Both dates use the same Tier 1-2 forest definition, through `forest_mask_2014` and
`forest_mask_2024` in `common.py`.

**Decisions locked.**

- `A2 = A1 - loss_ha`, gross loss of 2014 forest. Forest gain on non forest 2014 land is
  excluded, so the rate stays consistent with the reported loss.
- No comparison against a national rate. The component reports the site on its own terms. The
  national lookup, its CSV and the similar band threshold have been removed from the tool.
- The rate is shown to one decimal. Screening precision, not a monitoring figure.

**Puyravaud (2003).** `rate = (1 / (t2 - t1)) * ln(A2 / A1) * 100`

**Two edge cases decided here, not in the spec.**

1. No loss at all. The Puyravaud formula returns 0, which is correct but reads oddly as "an
   average of 0.0% per year", so the component uses a dedicated sentence instead.
2. Total loss, `A2 == 0`. All 2014 forest is gone and `ln(0)` is undefined. The component
   reports `rate_pct = None` and says the area lost all of its forest, rather than emitting an
   infinite rate.

**Example render.**

> Between 2014 and 2024, the project area lost 180 ha of forest, an average of 1.6% per year.

**Downstream use.** Observed loss is the empirical basis of the trajectory (Axis 2) and a direct
threat intensity signal. The AUD pathway signal now comes from the modelled risk in 1.6 rather
than from a national rate comparison here.

In [7]:
def analyze_historical_deforestation(aoi: AOI) -> ComponentResult:
    """Component 1.5. Forest loss 2014 to 2024 and its annual rate."""
    f2014 = forest_mask_2014(aoi)
    f2024 = forest_mask_2024(aoi)

    if f2014.is_empty:
        return not_applicable(
            "1.5 Historical Deforestation",
            "No forest was present in this project area in 2014, so a deforestation rate "
            "cannot be calculated.",
        )

    # Derived output rasters: forest at each date and the gross-loss grid.
    _loss = f2014.mask & ~f2024.mask
    loss_slice = RasterSlice(
        np.ma.masked_array(_loss.astype("float32"), mask=False),
        f2014.pixel_area_ha, f2014.transform, f2014.crs,
    )
    forest_rasters = {
        "1.5_forest_2014": f2014.as_slice(),
        "1.5_forest_2024": f2024.as_slice(),
        "1.5_forest_loss": loss_slice,
    }

    a1 = f2014.area_ha
    # Gross loss of 2014 forest. Gain elsewhere is deliberately not netted off.
    loss_ha = int((f2014.mask & ~f2024.mask).sum()) * f2014.pixel_area_ha
    a2 = a1 - loss_ha

    if loss_ha <= 0:
        rate_pct = 0.0
        narrative = "No forest loss was detected in this project area between 2014 and 2024."
    elif a2 <= 0:
        rate_pct = None  # ln(0) guard, see the note above
        narrative = (
            f"Between 2014 and 2024, the project area lost all of its {fmt_ha(a1)} of forest."
        )
    else:
        rate_pct = abs((1.0 / DEFOR_PERIOD_YEARS) * math.log(a2 / a1) * 100.0)
        narrative = (
            f"Between 2014 and 2024, the project area lost {fmt_ha(loss_ha)} of forest, an "
            f"average of {rate_pct:.1f}% per year."
        )

    return ComponentResult(
        component="1.5 Historical Deforestation",
        applicable=True,
        narrative=narrative,
        tables={},
        values={
            "forest_2014_ha": a1,
            "forest_2024_ha": a2,
            "loss_ha": loss_ha,
            "rate_pct": rate_pct,
        },
        rasters=forest_rasters,
    )


results["1.5"] = analyze_historical_deforestation(aoi)
show_result(results["1.5"])

[1.5 Historical Deforestation]
  Between 2014 and 2024, the project area lost 10,480 ha of forest, an average of 2.4% per year.


{'forest_2014_ha': 48826.26032492706,
 'forest_2024_ha': 38345.89668414915,
 'loss_ha': 10480.36364077791,
 'rate_pct': 2.4162076394938192}

  saved raster: D:\NBSTOOLV3\OUTPUTS\aoi1\rasters\1.5_forest_2014.tif
  saved raster: D:\NBSTOOLV3\OUTPUTS\aoi1\rasters\1.5_forest_2024.tif
  saved raster: D:\NBSTOOLV3\OUTPUTS\aoi1\rasters\1.5_forest_loss.tif


---
## 1.6 Deforestation Risk

Reports how the deforestation risk of the AOI forest compares to the national forest, as a
comparative narrative. No chart, text only.

**Data.** `prob.tif`, forestatrisk model output, UInt16 where 0 to 65535 encodes a 0 to 100
relative risk score. **The raster is already masked to forest upstream**, so its valid pixels
inside the AOI are the forest to assess. This component does not build its own forest mask and
never reads `LC2024.tif`.

**Interpretation warning.** The forestatrisk value is a relative spatial ranking, not an
absolute probability. Two properties of the model make this so: it is fitted with case control
sampling, which ties the intercept to the chosen sampling ratio rather than the true base rate,
and any predicted probability is conditional on the length of the calibration period. What
survives as valid information is the ordering of pixels, not the level. The tool therefore never
states an absolute chance of deforestation. It only places the AOI forest inside the national
distribution, as a percentile.

**Decisions locked.**

- Forest extent comes from `prob.tif` itself. Valid pixels are forest, masked pixels are not.
- AOI summary = median risk, robust to the skewed risk distribution: most forest is low risk and
  a small frontier is very high risk, so a mean would be pulled by the right tail.
- Comparison from the national percentile position: above p60 is "higher than", p40 to p60 is
  "similar to", below p40 is "lower than". No arbitrary band.
- Baseline is national, built from the same `prob.tif`, per country.
- Resampling is nearest. Bilinear would blend risk values across the forest boundary and
  contaminate the median.

**Consistency check to run once.** The forest that `prob.tif` was masked to must be the same
Tier 1-2 forest 2024 used by 1.5 and 2.1 (`FOREST_CODES = [1, 6, 7, 8, 10]`). If the upstream
mask used a different date or a different forest definition, this component silently reports on
a different area than the rest of the module.

**Open items, carried forward.**

1. `_national_percentile` clamps instead of extrapolating. The reference CSV holds p10 to p90
   only, so an AOI above p90 always reads as "top 10%" even when it is top 2%. This understates
   exactly the sites that matter most for AUD. Fix by adding p95 and p99 to the reference CSV.
2. The median hides the frontier. An AOI that is 80% safe interior and 20% active logging edge
   reads as low risk, although that 20% is what gives an AUD project its baseline.
3. An AOI median is compared against a table of pixel percentiles. Medians of areas cluster
   toward the centre more than individual pixels do, so "similar to the national average" fires
   more often than one in five sites.
4. If `prob.tif` is a mosaic of separately fitted regional models, the 0 to 100 scale is not
   guaranteed to be comparable across regions, and a national percentile table pools models with
   different calibrations.

**Example render.**

> Forest in this area is at higher deforestation risk than the national average, ranking in the
> top 36% of the country's forest for deforestation risk.

**Downstream use.** Standing forest at high risk is the core AUD signal.

In [8]:
def _national_percentile(value: float, breakpoints: dict[int, float]) -> float:
    """Position `value` inside a country's percentile breakpoints, by linear interpolation.

    Values outside p10 to p90 are clamped, not extrapolated, because the tail shape is unknown.
    Worked example. With p60 = 38 and p70 = 47, a value of 42 sits (42 - 38) / (47 - 38) = 0.44
    of the way between them, so the percentile is 60 + 0.44 * 10 = 64.4.
    """
    pcts = sorted(breakpoints)
    xs = [breakpoints[p] for p in pcts]
    return float(np.interp(value, xs, pcts))


def analyze_deforestation_risk(aoi: AOI, dominant_country: str | None) -> ComponentResult:
    """Component 1.6. Relative deforestation risk of the AOI forest against the national forest."""
    # prob.tif is already masked to forest upstream, so its valid pixels are the forest to
    # assess. No forest mask is built here and LC2024 is not read.
    prob = load_raster_clipped(PROB_RASTER, aoi, resampling="nearest")
    valid = prob.values.compressed()

    if valid.size == 0:
        return not_applicable(
            "1.6 Deforestation Risk",
            "No forest covered by the deforestation risk model is present in this project "
            "area, so deforestation risk cannot be assessed.",
        )

    # UInt16 storage: 0 to PROB_SCALE_MAX encodes a 0 to 100 relative risk score.
    forest_risk = valid.astype(float) / PROB_SCALE_MAX * 100.0
    aoi_risk = float(np.median(forest_risk))
    assessed_ha = valid.size * prob.pixel_area_ha

    if not dominant_country:
        return ComponentResult(
            component="1.6 Deforestation Risk",
            applicable=False,
            narrative="No country could be determined, so risk cannot be compared nationally.",
            values={"aoi_risk": aoi_risk, "assessed_ha": assessed_ha},
        )

    breakpoints = load_national_forest_risk_percentiles(dominant_country)
    if breakpoints is None:
        return ComponentResult(
            component="1.6 Deforestation Risk",
            applicable=False,
            narrative=(
                f"No national forest risk reference is available for {dominant_country}, so "
                "risk cannot be compared nationally."
            ),
            values={"aoi_risk": aoi_risk, "assessed_ha": assessed_ha},
            flags=[f"1.6: missing national risk reference for {dominant_country}."],
        )

    percentile = _national_percentile(aoi_risk, breakpoints)
    top_share = 100.0 - percentile

    if percentile > RISK_HIGHER_PCTL:
        comparison = "higher than"
        narrative = (
            "Forest in this area is at higher deforestation risk than the national average, "
            f"ranking in the top {fmt_pct(top_share)} of the country's forest for "
            "deforestation risk."
        )
    elif percentile < RISK_LOWER_PCTL:
        comparison = "lower than"
        narrative = (
            "Forest in this area is at lower deforestation risk than the national average, in "
            f"the bottom {fmt_pct(percentile)} of the country's forest."
        )
    else:
        comparison = "similar to"
        narrative = (
            "Forest in this area is at deforestation risk similar to the national average, "
            "around the national median."
        )

    return ComponentResult(
        component="1.6 Deforestation Risk",
        applicable=True,
        narrative=narrative,
        tables={},
        values={
            "aoi_risk": aoi_risk,
            "assessed_ha": assessed_ha,
            "national_percentile": percentile,
            "comparison": comparison,
        },
    )


country = results["1.2"].values.get("dominant_country")
results["1.6"] = analyze_deforestation_risk(aoi, country)
show_result(results["1.6"])

[1.6 Deforestation Risk]
  Forest in this area is at higher deforestation risk than the national average, ranking in the top 31% of the country's forest for deforestation risk.


{'aoi_risk': 19.35128220488098,
 'assessed_ha': 40765.179872189,
 'national_percentile': 69.19322135394907,
 'comparison': 'higher than'}

---
## 1.7 Natural Disaster Risks

Reports the natural disaster risks the AOI is exposed to, one card per risk, each with a
representative level. The narrative is a lead-in; the risks and their levels are the table.

**Data.** Five pre-classified risk rasters (`RISK_RASTERS`): cyclone, drought, fire, flood,
landslide. Encoding is 4-class, 1 = Very Low to 4 = High, with 0 as nodata.

**These are risk, not bare hazard.** Unlike a raw hazard map, these layers already fold in
exposure and vulnerability upstream, so the component reports risk directly. The five layers sit
at very different native resolutions (flood and landslide about 100 m, fire about 1 km, cyclone
about 11 km, drought about 28 km). For a small AOI the coarse layers (cyclone, drought) may fall
inside a single cell, giving a one-class distribution. That is expected.

**Decisions locked.**

- List any present risk. Every risk layer that has data over the AOI is listed, with its
  representative level, so nothing that is mapped is hidden.
- Representative level is conservative: the highest class covering at least 20 percent of the AOI
  valid risk area. In risk screening a false negative costs more than a false positive.
- No composite risk index. The five risks are not commensurable.

**Example render.**

> The selected area is susceptible to several natural disaster risks, including:

(the flood, landslide, fire, cyclone and drought cards, each with its level, follow as the table.)

**Downstream use.** Each risk level is read twice later: as permanence risk sensitivity, which
constrains activity design and durability, and as a disaster risk reduction co-benefit mapped to
the Triple Win pillars.

In [9]:
@dataclass(frozen=True)
class RiskCard:
    risk: str
    level_code: int | None
    level_label: str
    distribution: list[ClassShare]


def _representative_level(rows: list[ClassShare]) -> int | None:
    """Highest risk class covering at least RISK_PRESENCE_PCT of the valid area.

    At least one class always qualifies when data exists: the shares sum to 100 over the classes
    present, so some class must reach 20 percent.
    """
    qualifying = [r.code for r in rows if r.pct >= RISK_PRESENCE_PCT]
    return max(qualifying) if qualifying else None


def analyze_natural_risk(aoi: AOI) -> ComponentResult:
    """Component 1.7. One card per natural disaster risk, with a representative level each."""
    cards: list[RiskCard] = []
    risk_rasters: dict = {}

    for risk, path in RISK_RASTERS.items():
        raster = load_raster_clipped(path, aoi, resampling="nearest")
        if raster.valid_area_ha <= 0:
            # No coverage over this AOI: the risk is not "present", so it is left out of the list.
            continue
        risk_rasters[f"1.7_risk_{risk}"] = raster
        rows = tabulate_classes(raster, RISK_LEVELS, denominator_ha=raster.valid_area_ha)
        code = _representative_level(rows)
        cards.append(
            RiskCard(
                risk=risk,
                level_code=code,
                level_label=RISK_LEVELS[code] if code else "No data",
                distribution=rows,
            )
        )

    # Any present risk is listed, ordered by level (highest first), then name for ties.
    present = [c for c in cards if c.level_code]
    present.sort(key=lambda c: (-c.level_code, c.risk))

    if present:
        narrative = "The selected area is susceptible to several natural disaster risks, including:"
    else:
        narrative = "The selected area has no natural disaster risk data for this location."

    return ComponentResult(
        component="1.7 Natural Disaster Risks",
        applicable=bool(present),
        narrative=narrative,
        tables={"risk_cards": present},
        values={c.risk: c.level_code for c in present},
        rasters=risk_rasters,
    )


results["1.7"] = analyze_natural_risk(aoi)
show_result(results["1.7"])

[1.7 Natural Disaster Risks]
  The selected area is susceptible to several natural disaster risks, including:
  risk_cards:


,risk,level_code,level_label,distribution
0,drought,3,Moderate,"[{'code': 1, 'label': 'Very Low', 'area_ha': 0..."
1,cyclone,1,Very Low,"[{'code': 1, 'label': 'Very Low', 'area_ha': 7..."
2,fire,1,Very Low,"[{'code': 1, 'label': 'Very Low', 'area_ha': 4..."
3,flood,1,Very Low,"[{'code': 1, 'label': 'Very Low', 'area_ha': 7..."
4,landslide,1,Very Low,"[{'code': 1, 'label': 'Very Low', 'area_ha': 5..."


  saved table: D:\NBSTOOLV3\OUTPUTS\aoi1\tables\1.7_risk_cards.csv


{'drought': 3, 'cyclone': 1, 'fire': 1, 'flood': 1, 'landslide': 1}

  saved raster: D:\NBSTOOLV3\OUTPUTS\aoi1\rasters\1.7_risk_cyclone.tif
  saved raster: D:\NBSTOOLV3\OUTPUTS\aoi1\rasters\1.7_risk_drought.tif
  saved raster: D:\NBSTOOLV3\OUTPUTS\aoi1\rasters\1.7_risk_fire.tif
  saved raster: D:\NBSTOOLV3\OUTPUTS\aoi1\rasters\1.7_risk_flood.tif
  saved raster: D:\NBSTOOLV3\OUTPUTS\aoi1\rasters\1.7_risk_landslide.tif


---
## 1.8 Land Cover

Land cover composition of the project area, from the 2024 20-class map (`LC2024_RASTER`).

Two tables: the **full** distribution (every class present, with area and share of the AOI,
summing to 100 via a No-data/other row), and a **subset** of the six largest land cover classes.
Denominator is the total AOI area, so each share is a share of the site. The AOI-clipped land
cover raster is saved.

**Narrative.** "The major land cover categories in the selected area are:" leads into the
six-class subset table.

In [10]:
LC_NODATA_LABEL = "No data / other"


def analyze_land_cover(aoi: AOI) -> ComponentResult:
    """Component 1.8. Land cover composition of the AOI, from the 2024 20-class map."""
    lc = load_raster_clipped(LC2024_RASTER, aoi, resampling="nearest")
    if lc.valid_area_ha <= 0:
        return not_applicable(
            "1.8 Land Cover", "No land cover data is available for this project area."
        )

    # Denominator is the whole site; a No-data/other row absorbs code 0 so the full table sums
    # to 100. Snow, Water and Other land are kept as their own classes, not folded into No-data.
    rows = tabulate_classes(lc, LC2024_CLASSES, denominator_ha=aoi.area_ha)
    mapped_ha = sum(r.area_ha for r in rows)
    other_ha = max(0.0, aoi.area_ha - mapped_ha)
    rows.append(ClassShare(code=0, label=LC_NODATA_LABEL, area_ha=other_ha,
                           pct=safe_pct(other_ha, aoi.area_ha)))

    full = sort_by_area([r for r in rows if r.area_ha > 0])              # full table, sums to 100
    top6 = sort_by_area([r for r in full if r.code != 0])[:LC_TOP_N]     # six largest LC classes
    dom = top6[0] if top6 else None

    return ComponentResult(
        component="1.8 Land Cover",
        applicable=True,
        narrative="The major land cover categories in the selected area are:",
        tables={"land_cover_full": full, "land_cover_top6": top6},
        values={
            "chart_series": "land_cover_full",
            "chart_unit": "%",
            "chart_axis_label": "Share of project area (%)",
            "dominant_class": dom.label if dom else None,
            "class_count": sum(1 for r in full if r.code != 0),
        },
        rasters={"1.8_land_cover": lc},
    )


results["1.8"] = analyze_land_cover(aoi)
show_result(results["1.8"])

[1.8 Land Cover]
  The major land cover categories in the selected area are:
  land_cover_full:


,code,label,area_ha,pct
0,8,Evergreen Forest,39985.706108,59.291647
1,9,Shrubland,13823.395165,20.497621
2,3,Palm,5488.379795,8.138285
3,16,Cropland,5324.291961,7.894972
4,19,Bareland,875.571980,1.298317
5,20,Other land,808.119473,1.198297
6,10,Mixed forest,640.449318,0.949672
7,15,Building,197.464592,0.292805
8,1,Flooded forest,129.837338,0.192526
9,18,Wetland,52.686349,0.078124


  saved table: D:\NBSTOOLV3\OUTPUTS\aoi1\tables\1.8_land_cover_full.csv
  land_cover_top6:


,code,label,area_ha,pct
0,8,Evergreen Forest,39985.706108,59.291647
1,9,Shrubland,13823.395165,20.497621
2,3,Palm,5488.379795,8.138285
3,16,Cropland,5324.291961,7.894972
4,19,Bareland,875.571980,1.298317
5,20,Other land,808.119473,1.198297


  saved table: D:\NBSTOOLV3\OUTPUTS\aoi1\tables\1.8_land_cover_top6.csv


{'chart_series': 'land_cover_full',
 'chart_unit': '%',
 'chart_axis_label': 'Share of project area (%)',
 'dominant_class': 'Evergreen Forest',
 'class_count': 12}

  saved raster: D:\NBSTOOLV3\OUTPUTS\aoi1\rasters\1.8_land_cover.tif


---
## Save

Each section above already ran and displayed itself. This cell writes them to one combined JSON.

In [11]:
path = save_results(results, aoi, aoi_id, STAGE_GENERAL)
print(f"Saved {path}")

Saved D:\NBSTOOLV3\OUTPUTS\aoi1\aoi1__F02-P2-general.json
